# Preprocesamiento

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import RobustScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

## Importación de los datos

In [5]:
ruta = Path("..") / "data" / "raw" /"train.csv"
train = pd.read_csv(ruta)

Explicar que la mayoria de datos viene muy limpio del EDA Y que no es necesario un one hoting o hacer muchos cambios al dataset porque por el pca no hay multicoliedad. Presentamos outliers

## Feature Engineering 

In [6]:
X_train = train.drop(columns="Class")
y_train = train["Class"]

def crear_variables(X):
    X = X.copy()
    X["Hour_sin"] = np.sin(2 * np.pi * X["Hour"] / 24)
    X["Hour_cos"] = np.cos(2 * np.pi * X["Hour"] / 24)
    return X.drop(columns=["Time", "Hour"])

### Transformaciones y Escalado

In [ ]:
amount_pipe = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", RobustScaler())
])

columnas = ColumnTransformer(
    [("amount", amount_pipe, ["Amount"])],
    remainder="passthrough", 
    verbose_feature_names_out=False,
).set_output(transform="pandas")

preprocesador = Pipeline([
    ("variables", FunctionTransformer(crear_variables)),
    ("columnas", columnas),
])

In [9]:
X_train_clean = preprocesador.fit_transform(X_train)
train_clean = X_train_clean.assign(Class=y_train)

assert len(train_clean) == len(train)
assert train_clean.isnull().sum().sum() == 0

print(train_clean.shape)
train_clean.head()

(226980, 32)


,Amount,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V22,V23,V24,V25,V26,V27,V28,Hour_sin,Hour_cos,Class
0,0.144933,2.238954,-1.724499,-2.151484,-2.577803,0.993668,3.565492,-1.785957,0.860122,-1.264003,...,-0.049333,0.278442,0.684735,-0.219028,-0.159167,0.037920,-0.049932,-0.866025,-0.500000,0
1,-0.429998,-1.315062,1.630783,0.597001,-0.038359,-0.404580,-0.965712,0.212249,0.735381,-1.267926,...,-0.946773,0.323904,0.515632,-0.713000,-0.266503,-0.017794,0.051058,-0.500000,0.866025,0
2,-0.149784,1.908801,0.021184,-2.087997,0.129310,1.161468,0.605244,-0.022371,0.180296,0.283819,...,1.095842,-0.044874,-1.689517,0.106098,0.007758,0.045164,-0.053068,0.258819,-0.965926,0
3,-0.094070,1.811257,0.316556,0.316751,3.880231,0.048454,1.020163,-0.734868,0.233651,0.681423,...,0.700422,0.174064,0.702997,-0.212523,-0.010018,-0.017740,-0.038006,0.258819,0.965926,0
4,0.032392,1.358817,-1.120881,0.550266,-1.547659,-1.194950,0.275448,-1.201843,0.212889,-2.094285,...,-0.636442,0.252758,-0.344160,-0.064282,-0.439622,0.062524,0.013095,0.866025,-0.500000,0


### Remuestreo

## Exportación df limpio para modelado

In [ ]:
ruta = Path("..") / "data" / "cleaned"

train_clean.to_csv(ruta/"train_clean.csv", index=False)